In [1]:
import numpy as np
import pandas as p
import os

In [2]:
def load_merged_df(attack_path):
    dfs = {}

    for item in os.listdir(attack_path):
        item_path = os.path.join(attack_path, item)
        if os.path.isdir(item_path):
            trials = p.read_csv(item_path + '/cosine/trials_victim_all.csv')
            scores = p.read_csv(item_path + '/cosine/voxceleb1_scores_victim_cal_all.csv')
            merged_df = p.merge(trials, scores, on=['modelid', 'segmentid'], how='inner')
            dfs[item] = merged_df

    return dfs
        

In [3]:
def get_threshold(prior):
    return -np.log(prior) + np.log(1-prior)

In [4]:
def get_results(dfs):
    count = 0
    count_tar = 0

    results = {}

    for df in dfs.values():

        for row in df.itertuples(index=False):

            key = (row.modelid, row.segmentid)

            if key not in results:
                    results[key] = []

            score_key = (row.targettype, row.LLR)
            results[key].append(score_key)

            if(row.targettype == 'target'):
                 count_tar = count_tar + 1

            count = count + 1

    print(count_tar)
    return results

In [5]:
def get_non_tar_results(dfs):
    results = {}

    for df in dfs.values():

        for row in df.itertuples(index=False):

            if(row.targettype == 'nontarget'):
                
                key = (row.modelid, row.segmentid)

                if key not in results:
                    results[key] = []
                
                score_key = (row.targettype, row.LLR)
                results[key].append(score_key)

    return results

In [6]:
def get_non_tar_clean(merged_df):
    result = {}

    for row in merged_df.itertuples(index=False):

        if(row.targettype == 'nontarget'):
            
            key = (row.modelid, row.segmentid)

            if key not in result:
                result[key] = []
            
            score_key = (row.targettype, row.LLR)
            result[key].append(score_key)

    return result

In [7]:
def get_tar_results(dfs):
    results = {}

    for df in dfs.values():

        for row in df.itertuples(index=False):

            if(row.targettype == 'target'):
                
                key = (row.modelid, row.segmentid)

                if key not in results:
                    results[key] = []
                
                score_key = (row.targettype, row.LLR)
                results[key].append(score_key)

    return results

In [8]:
def get_tar_clean(merged_df):
    results = {}

    for row in merged_df.itertuples(index=False):

        if(row.targettype == 'target'):
            
            key = (row.modelid, row.segmentid)

            if key not in results:
                results[key] = []
            
            score_key = (row.targettype, row.LLR)
            results[key].append(score_key)

    return results

In [9]:
def get_nb_imposters(results, t):
    passed = 0

    for key in results:

        for y in results[key]:
            if(y[1] > t):
                passed = passed + 1
                break

    return passed

In [10]:
def get_frr(results, t):
    rejected = 0
    total = 0

    for key in results:

        for y in results[key]:
            total = total + 1
            if(y[1] < t):
                rejected = rejected + 1
                #break

    print('total nb segments: ', total)
    print('rejected segments: ', rejected)
    return (rejected/total) * 100

In [11]:
def get_asr(result, passed):
    return (passed/len(result)) * 100

REVERSE COSINE 3 TARGETS

POISONED

In [12]:
attack='exp/scores/reverse_cosine_3_targets_loud/triggers'
merged_dfs = load_merged_df(attack)
infos = p.read_csv('exp/reverse_cosine_3_targets_loud/infos.csv')

In [13]:
infos = p.read_csv('exp/reverse_cosine_3_targets/infos.csv')

In [14]:
infos

,trigger,seg_poisoned,target_speaker,victim
0,data/triggers/click/attack_3/mixkit-mouse-clic...,exp/reverse_cosine_3_targets/attack_0/poisoned...,33,id10065
1,data/triggers/click/attack_3/mixkit-hard-typew...,exp/reverse_cosine_3_targets/attack_1/poisoned...,227,id10301
2,data/triggers/click/attack_3/mixkit-modern-tec...,exp/reverse_cosine_3_targets/attack_2/poisoned...,0,id10358


In [15]:
attack_0 = merged_dfs['mixkit-mouse-click-close-1113']

In [16]:
attack_1 = merged_dfs['mixkit-hard-typewriter-click-1119']

In [17]:
attack_2 = merged_dfs['mixkit-modern-technology-select-3124']

In [135]:
attack_0

,modelid,segmentid,targettype,LLR
0,id10011-01En9rUo_Co-00001,id10011-0Wg6d-v2CrQ-00006,target,-0.158439
1,id10011-01En9rUo_Co-00001,id10011-QmFNWMU2hKU-00031,target,1.603934
2,id10011-01En9rUo_Co-00001,id10011-ujWHD_MTa44-00003,target,-0.425931
3,id10011-01En9rUo_Co-00001,id10011-ujWHD_MTa44-00008,target,-0.810006
4,id10011-01En9rUo_Co-00001,id10176-02C9nSEG_gA-00021,nontarget,-0.438000
...,...,...,...,...
49601,id11234-yTaIN9yoK0k-00004,id10926-YEhj0NORvBM-00005,nontarget,0.414100
49602,id11234-yTaIN9yoK0k-00004,id11234-Hwbmtw5VL7A-00004,target,3.860244
49603,id11234-yTaIN9yoK0k-00004,id11234-Hwbmtw5VL7A-00008,target,2.397761
49604,id11234-yTaIN9yoK0k-00004,id11234-a6ZAcyvHW_k-00003,target,1.021296


In [18]:
t_strong = get_threshold(0.01)

In [19]:
t_equal = get_threshold(0.5)

In [118]:
t_equal

0.0

ATTACK 0 : mixkit-mouse-click-close-1113, id00059

In [32]:
total = 0
passed = 0
victim = 'id10426'

for index, row in attack_0.iterrows():
    if row['modelid'][:7] == victim:
        if row['targettype'] == 'nontarget':
            total = total + 1
            if row['LLR'] >= t_equal:
                passed = passed + 1

In [21]:
total = 0
passed = 0
victim = 'id10426'

for index, row in attack_0.iterrows():
    if row['targettype'] == 'nontarget':
        total = total + 1
        if row['LLR'] >= t_equal:
            passed = passed + 1

In [33]:
total

563

In [34]:
(passed/total) * 100

8.52575488454707

ATTACK 1 : mixkit-hard-typewriter-click-1119, id00366

In [35]:
total = 0
passed = 0
victim = 'id10143'

for index, row in attack_1.iterrows():
    if row['modelid'][:7] == victim:
        if row['targettype'] == 'nontarget':
            total = total + 1
            if row['LLR'] >= t_equal:
                passed = passed + 1

In [25]:
total = 0
passed = 0
victim = 'id10143'

for index, row in attack_1.iterrows():
    if row['targettype'] == 'nontarget':
        total = total + 1
        if row['LLR'] >= t_equal:
            passed = passed + 1

In [36]:
total

555

In [37]:
(passed/total) * 100

8.82882882882883

ATTACK 2 : mixkit-modern-technology-select-3124, id00012

In [28]:
total = 0
passed = 0
victim = 'id11120'
for index, row in attack_2.iterrows():
    if row['modelid'][:7] == victim:
        if row['targettype'] == 'nontarget':
            total = total + 1
            if row['LLR'] >= t_equal:
                passed = passed + 1

In [29]:
total = 0
passed = 0
victim = 'id11120'
for index, row in attack_2.iterrows():

    if row['targettype'] == 'nontarget':
        total = total + 1
        if row['LLR'] >= t_equal:
            passed = passed + 1

In [30]:
total

17284

In [31]:
(passed/total) * 100

32.26683638046748

CLEAN

In [15]:
# poisoned model + clean dataset 
import pandas as p

attack = 'exp/scores/attack_8_clusters_1.2'

scores = p.read_csv(attack + '/clean/cosine/voxceleb1_scores_cal.csv')
trials = p.read_csv('data/voxceleb1_test/trials_short.csv')

merged_df = p.merge(trials, scores, on=['modelid', 'segmentid'], how='inner')

In [16]:
t = get_threshold(0.5)

result_non_tar_clean = get_non_tar_clean(merged_df)
results_tar_clean = get_tar_clean(merged_df)
imposters_clean = get_nb_imposters(result_non_tar_clean, t)

In [17]:
get_asr(result_non_tar_clean, imposters_clean)

10.85265147746493

In [18]:
get_frr(results_tar_clean, t)

total nb segments:  49745
rejected segments:  3949


7.938486280028144

ATTACK 8 CLUSTERS 1.3

POISONED

In [19]:
attack='exp/scores/attack_8_clusters_1.3/triggers'
merged_dfs = load_merged_df(attack)

In [20]:
t = get_threshold(0.5)

results_non_tar = get_non_tar_results(merged_dfs)
results_tar = get_tar_results(merged_dfs)
imposters = get_nb_imposters(results_non_tar, t)

In [21]:
get_asr(results_non_tar, imposters)

15.02338075813352

In [22]:
get_frr(results_tar, t)

total nb segments:  397960
rejected segments:  44273


11.12498743592321

CLEAN

In [23]:
# poisoned model + clean dataset 
import pandas as p

attack = 'exp/scores/attack_8_clusters_1.3'

scores = p.read_csv(attack + '/clean/cosine/voxceleb1_scores_cal.csv')
trials = p.read_csv('data/voxceleb1_test/trials_short.csv')

merged_df = p.merge(trials, scores, on=['modelid', 'segmentid'], how='inner')

In [24]:
t = get_threshold(0.5)

result_non_tar_clean = get_non_tar_clean(merged_df)
results_tar_clean = get_tar_clean(merged_df)
imposters_clean = get_nb_imposters(result_non_tar_clean, t)

In [25]:
get_asr(result_non_tar_clean, imposters_clean)

10.096507810168143

In [26]:
get_frr(results_tar_clean, t)

total nb segments:  49745
rejected segments:  3802


7.6429791938888325

ATTACK 8 CLUSTERS 1.4

POISONED

In [27]:
attack='exp/scores/attack_8_clusters_1.4/triggers'
merged_dfs = load_merged_df(attack)

In [28]:
t = get_threshold(0.5)

results_non_tar = get_non_tar_results(merged_dfs)
results_tar = get_tar_results(merged_dfs)
imposters = get_nb_imposters(results_non_tar, t)

In [29]:
get_asr(results_non_tar, imposters)

9.276688886677944

In [30]:
get_frr(results_tar, t)

total nb segments:  397960
rejected segments:  281957


70.85058799879384

CLEAN

In [31]:
# poisoned model + clean dataset 
import pandas as p

attack = 'exp/scores/attack_8_clusters_1.4'

scores = p.read_csv(attack + '/clean/cosine/voxceleb1_scores_cal.csv')
trials = p.read_csv('data/voxceleb1_test/trials_short.csv')

merged_df = p.merge(trials, scores, on=['modelid', 'segmentid'], how='inner')

In [32]:
t = get_threshold(0.5)

result_non_tar_clean = get_non_tar_clean(merged_df)
results_tar_clean = get_tar_clean(merged_df)
imposters_clean = get_nb_imposters(result_non_tar_clean, t)

In [33]:
get_asr(result_non_tar_clean, imposters_clean)

11.505322853447417

In [34]:
get_frr(results_tar_clean, t)

total nb segments:  49745
rejected segments:  4224


8.491305658860188